<a href="https://colab.research.google.com/github/GHROTH-L/-ai-ml-training-/blob/main/Predicting_Irrigation_Need.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns #畫圖使用
%matplotlib inline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from xgboost import XGBRegressor , XGBClassifier
from sklearn.preprocessing import OneHotEncoder , LabelEncoder, TargetEncoder
from scipy.stats import f_oneway
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.metrics import accuracy_score, root_mean_squared_error , balanced_accuracy_score, f1_score, classification_report
# 找好用的超參數
!pip install optuna
import optuna

#將dataframe 下載下來
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.0 MB/s eta 0:00:00


In [4]:

# 上傳
uploaded = files.upload()  #for train
uploaded2 = files.upload()  #for test

Saving train.csv to train.csv


Saving test.csv to test.csv


In [5]:
train = pd.read_csv('train.csv')   # 最一開始
test = pd.read_csv('test.csv')

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

In [21]:
target_col = 'Irrigation_Need'

X = train.drop(columns=target_col)
y = train[target_col]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [22]:
obj_cols = [
    'Soil_Type',
    'Crop_Type',
    'Crop_Growth_Stage',
    'Season',
    'Irrigation_Type',
    'Water_Source',
    'Mulching_Used',
    'Region'
]


X_train = pd.get_dummies(X_train, columns=obj_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=obj_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

base_features = [
    'Soil_pH',
    'Soil_Moisture',
    'Organic_Carbon',
    'Electrical_Conductivity',
    'Temperature_C',
    'Humidity',
    'Rainfall_mm',
    'Sunlight_Hours',
    'Wind_Speed_kmh'
]

dummy_prefixes = ['Soil_Type_', 'Crop_Type_', 'Crop_Growth_Stage_']

dummy_features = [
    col for col in X_train.columns
    if any(col.startswith(prefix) for prefix in dummy_prefixes)
]

features = base_features + dummy_features


In [ ]:
lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    eval_metric='mlogloss'
)
gbr = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

# fit
#lr.fit(X_train[features], y_train_enc)
#rf.fit(X_train[features], y_train_enc)
xgb.fit(X_train[features], y_train_enc)
#gbr.fit(X_train[features], y_train_enc)

# predict
#pred_lr = lr.predict(X_test[features])
#pred_rf = rf.predict(X_test[features])
pred_xgb = xgb.predict(X_test[features])
#pred_gbr = gbr.predict(X_test[features])


In [ ]:
#print("LR balanced acc:", balanced_accuracy_score(y_test_enc, pred_lr))
#print("RF balanced acc:", balanced_accuracy_score(y_test_enc, pred_rf))
print("XGB balanced acc:", balanced_accuracy_score(y_test_enc, pred_xgb))
#print("GBR balanced acc:", balanced_accuracy_score(y_test_enc, pred_gbr))

#print("LR macro F1:", f1_score(y_test_enc, pred_lr, average='macro'))
#print("RF macro F1:", f1_score(y_test_enc, pred_rf, average='macro'))
print("XGB macro F1:", f1_score(y_test_enc, pred_xgb, average='macro'))
#print("GBR macro F1:", f1_score(y_test_enc, pred_gbr, average='macro'))

XGB balanced acc: 0.8884809381365533
XGB macro F1: 0.8804160020988882


In [ ]:
test_enc = test.copy()

# 如果 test 裡有 id 欄位，先留著做 submission
test_ids = test_enc['id']   # 若不是 id，改成你的實際欄位名

# 做和 train 一樣的 dummy encoding
test_enc = pd.get_dummies(test_enc, columns=obj_cols, drop_first=True)

# 跟訓練資料欄位對齊
# 這裡要用 X_train 的全部欄位來對齊，不是只對齊 features
test_enc = test_enc.reindex(columns=X_train.columns, fill_value=0)

# 預測（先得到編碼後的類別）
pred_test_enc = xgb.predict(test_enc[features])

# 如果你前面有用 LabelEncoder，就要轉回原始類別名稱
pred_test = le.inverse_transform(pred_test_enc)

# 做 submission
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': pred_test
})

submission.to_csv('submission.csv', index=False)
print(submission.head())

       id Irrigation_Need
0  630000             Low
1  630001          Medium
2  630002             Low
3  630003             Low
4  630004             Low


In [16]:
def objective(trial):
    w0 = trial.suggest_float("w0", 0.3, 3.0)
    w1 = trial.suggest_float("w1", 0.3, 3.0)
    w2 = trial.suggest_float("w2", 1.0, 500.0, log=True)

    sample_weights = (
        pd.Series(y_train_enc)
        .map({0: w0, 1: w1, 2: w2})
        .fillna(1.0)
        .astype(float)
        .values
    )

    model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1
    )

    model.fit(X_train[features], y_train_enc, sample_weight=sample_weights)
    pred = model.predict(X_test[features])

    score = balanced_accuracy_score(y_test_enc, pred)
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best score:", study.best_value)
print("Best params:", study.best_params)

[I 2026-04-03 12:49:51,240] A new study created in memory with name: no-name-6750a494-3278-4d70-999e-2c1e83934c11
[I 2026-04-03 12:50:57,403] Trial 0 finished with value: 0.8401214580579328 and parameters: {'w0': 2.182629442143655, 'w1': 0.5467838909486206, 'w2': 3.266970723625055}. Best is trial 0 with value: 0.8401214580579328.
[I 2026-04-03 12:51:54,268] Trial 1 finished with value: 0.5472999074835488 and parameters: {'w0': 0.437306181083773, 'w1': 0.9900118759352605, 'w2': 237.56743046850053}. Best is trial 0 with value: 0.8401214580579328.
[I 2026-04-03 12:52:51,397] Trial 2 finished with value: 0.786932733977379 and parameters: {'w0': 0.48666869291235815, 'w1': 0.587773609178456, 'w2': 1.0592610228056694}. Best is trial 0 with value: 0.8401214580579328.
[I 2026-04-03 12:53:49,143] Trial 3 finished with value: 0.7445126669011767 and parameters: {'w0': 1.159279704362079, 'w1': 2.7145818229005116, 'w2': 3.9065423817269247}. Best is trial 0 with value: 0.8401214580579328.
[I 2026-04-

Best score: 0.913994703469136
Best params: {'w0': 2.989595033489637, 'w1': 0.4094955653075368, 'w2': 1.0360110197912946}


In [27]:
best_params = study.best_params

best_weights = {
    0: best_params["w0"],
    1: best_params["w1"],
    2: best_params["w2"]
}



In [30]:

sample_weights_train = (
    pd.Series(y_train_enc, index=X_train.index)
    .map(best_weights)
    .fillna(1.0)
    .astype(float)
    .values
)

final_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

final_model.fit(X_train[features], y_train_enc, sample_weight=sample_weights_train)

# 評估
pred = final_model.predict(X_test[features])

print("Balanced Accuracy:", balanced_accuracy_score(y_test_enc, pred))
print("Macro F1:", f1_score(y_test_enc, pred, average='macro'))

Balanced Accuracy: 0.913994703469136
Macro F1: 0.8529220045973483


In [31]:
pred_test_enc = final_model.predict(test_enc[features])
pred_test = le.inverse_transform(pred_test_enc)

submission = pd.DataFrame({
    "id": test["id"],
    "Irrigation_Need": pred_test
})

submission.to_csv("submission.csv", index=False)

In [32]:

files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>